In [1]:
pip install requests beautifulsoup4 pandas numpy

Note: you may need to restart the kernel to use updated packages.


In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

def get_kin_summary(keyword:str, page:int=1):
    url = f"https://kin.naver.com/search/list.naver?query={keyword}&page={page}"
    
    headers = {'User-Agent': 'Mozilla/5.0'}
    response = requests.get(url, headers=headers)
    
    links = []
    try:
        response.raise_for_status()
        
        soup = BeautifulSoup(response.text, 'lxml')
        
        a_tags = soup.select('ul.basic1 > li > dl > dt > a')
        for a in a_tags:
            links.append(a['href'])
            
    except Exception as e:
        print(f'{page}페이지 요약 목록 수집 중 예외 발생: {e}')
        
    return pd.DataFrame({'link': links})

def get_kin_detail(url:str):
    headers = {'User-Agent': 'Mozilla/5.0'}
    response = requests.get(url, headers=headers)

    try:
        response.raise_for_status()

        soup = BeautifulSoup(response.text.strip(), 'lxml')
        
        title_element = soup.select_one('.title') if soup.select_one('.title') else soup.select_one('.endTitleSection')
        if not title_element:
            return {}

  
        icon_element = title_element.select_one('.iconQuestion')
        if icon_element:
            icon_element.decompose()

        title = title_element.text.strip()

        content_element = soup.select_one('.questionDetail') or soup.select_one('.c-heading__content')
        content = content_element.text.strip() if content_element else "내용 없음"
        
        return {
            'title' : title,
            'content' : content,
        }

    except Exception as e:  
        print(f'상세 페이지 수집 중 예외 발생: {e} / URL: {url}')
        return {}

def get_kin_full(keyword:str, max_page:int=10):
    """지정된 페이지 수만큼 지식인 질문들을 수집하여 데이터프레임으로 반환하는 메인 함수"""
    data_list = []
    
    for page in range(1, max_page + 1):
        print(f"--- {page}페이지 수집 중 ---")
        
        df_summary = get_kin_summary(keyword, page)
        
        for link in df_summary['link']:
            dic = get_kin_detail(link)
            if dic:
                data_list.append(dic)
        
        time.sleep(1)
            
    return pd.DataFrame(data_list)

if __name__ == "__main__":
    keyword = "고민"
    
    print(f"[{keyword}] 키워드로 데이터 수집을 시작합니다...")

    result_df = get_kin_full(keyword, max_page=10)
    
    print(f"\n총 {len(result_df)}개의 데이터를 수집했습니다.")
    
    csv_filename = "고민정보_수집.csv"
    result_df.to_csv(csv_filename, index=False, encoding="utf-8-sig")
    
    print(f"데이터가 [{csv_filename}] 파일로 성공적으로 저장되었습니다.")

[고민] 키워드로 데이터 수집을 시작합니다...
--- 1페이지 수집 중 ---
--- 2페이지 수집 중 ---
--- 3페이지 수집 중 ---
--- 4페이지 수집 중 ---
--- 5페이지 수집 중 ---
--- 6페이지 수집 중 ---
--- 7페이지 수집 중 ---
--- 8페이지 수집 중 ---
--- 9페이지 수집 중 ---
--- 10페이지 수집 중 ---

총 100개의 데이터를 수집했습니다.
데이터가 [고민정보_수집.csv] 파일로 성공적으로 저장되었습니다.


In [4]:
import pandas as pd
import numpy as np

df = result_df.copy()
df_cleaned = df.dropna(subset=['title', 'content'])
df_final = df_cleaned[df_cleaned['content'].str.len() > 5].copy()

print("--- 전처리 결과 ---")
print(f"초기 수집 데이터 수: {len(df)}개")
print(f"결측치 제거 후 데이터 수: {len(df_cleaned)}개")
print(f"5자 이하 제거 후 최종 데이터 수: {len(df_final)}개")
print("\n[전처리 완료 데이터 샘플]")
print(df_final.head())

--- 전처리 결과 ---
초기 수집 데이터 수: 100개
결측치 제거 후 데이터 수: 100개
5자 이하 제거 후 최종 데이터 수: 100개

[전처리 완료 데이터 샘플]
   title                                            content
0  1개 답변  대구 소아/남 틱장애아이 진료를 보고 왔는데 약을 써보자는 이야기를 들었는데요. 그...
1  1개 답변  서울 10대 초반/여 성조숙증딸아이 검사 결과 성조숙증 초기라고 해서 성조숙증치료를...
2  1개 답변  대구 30대 중반/여 백반증지금은 병원에서 레이저 치료를 꾸준히 받고 있는 중인데,...
3  1개 답변  울산 50대 중반/남 여드름흉터여드름흉터가 많이 심한 편이라 이것 저것 치료 많이 ...
4  1개 답변  역삼역 50대 초반/남 전립선비대증 치료요즘 소변을 봐도 시원하지 않고 밤마다 화장...
